In [3]:
import logging
import time
from datetime import datetime
import warnings
import shutil
import json
import pickle
import torch
import sys

sys.path.append("/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src")

import os
import pandas as pd
import numpy as np
import networkx as nx
import scanpy as sc
import anndata as ad

import torch.nn as nn
from tqdm import tqdm
from collections import Counter
from itertools import islice

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

from models.DyGMamba import DyGMamba
from models.modules import MergeLayer, MergeLayerTD

from utils.load_configs import load_link_prediction_args

from utils.DataLoader import get_model_data
from utils.DataLoader import get_idx_data_loader
from utils.utils import get_neighbor_sampler, NegativeEdgeSampler
from utils.utils import get_parameter_sizes
from utils.utils import set_random_seed
from utils.utils import convert_to_gpu, create_optimizer
from utils.EarlyStopping import EarlyStopping
from utils.metrics import get_link_prediction_metrics
from models.evaluate_models_utils import evaluate_model_link_prediction
from models.inference_grn import model_link_prediction

# Configuration

In [9]:
import os
print("********************** start ********************")

start_time = time.time()  # start the time
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Start the job")

# get arguments
args = load_link_prediction_args(is_evaluation=False)

print("**********************device********************")
print(f"Now use device is {args.device}")

org_data_path = "/home/liyang/BioWuYan/dygmamba_project/data/original/"

dyg_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/dygmamba/res/result2/"
# os.makedirs(data_path, exist_ok = True)

********************** start ********************
[2025-12-23 11:05:23] Start the job
**********************device********************
Now use device is cuda:0


# Load Data

In [8]:


feat_path = dyg_result_path + "edge_features.npy"
edge_label_path = dyg_result_path + "edge_labels.npy"

Edge_feature = np.load(feat_path, mmap_mode="r")
Edge_feature = Edge_feature.reshape(-1,1).copy()
Edge_label = np.load(edge_label_path, mmap_mode="r")
Edge_label = Edge_label.reshape(-1,1).copy()

with open(dyg_result_path + "node_feature_data.pkl", "rb") as f:
    load_data = pickle.load(f)

Node_feature = load_data['node_feature']

Node_id = pd.read_pickle(dyg_result_path + "node_id.pkl")

graph_df = pd.read_pickle(dyg_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

# Result Analysis

## TF-region data

In [10]:
from data_preprocess import filter_jaspar_tf

jaspar_tf_region_file = org_data_path + "jaspar_data.h5ad"

jaspar_data = ad.read_h5ad(jaspar_tf_region_file)

adata_region_tf = filter_jaspar_tf(jaspar_data)

print(adata_region_tf)


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 1: 过滤 Peaks (行)
  > 找到 72563 / 72584 个 peaks 至少有 1 个 TF 结合。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 2: 过滤 TFs (列)
  > 找到 879 / 879 个 TFs 至少结合 1 个 peak。
  > 最终形状: (72563, 879)
AnnData object with n_obs × n_vars = 72563 × 879
    uns: 'description'


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [11]:
import pandas as pd

coo_matrix = adata_region_tf.X.tocoo()

# 创建一个 DataFrame 来存储 TF-Peak 的连接
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})
tf_peak_df


,Peak,TF,value
0,chr1-10231-10656,Dmrt1,1
1,chr1-10231-10656,ZNF708,1
2,chr1-10231-10656,CDX2,1
3,chr1-10231-10656,Lef1,1
4,chr1-10231-10656,SP1,1
...,...,...,...
55056385,chrY-11310224-11310574,Stat5a::Stat5b,1
55056386,chrY-11310224-11310574,GATA2,1
55056387,chrY-11310224-11310574,ETV2::FOXI1,1
55056388,chrY-11310224-11310574,Bhlha15,1


## Region-gene

In [12]:
graph_df = pd.read_pickle(dyg_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

result_graph = New_Graph.copy()

result_path = dyg_result_path + 'my_result_run{run}.npy'

predict_edge_label = np.load(result_path)

# binary_output = (predict_edge_label > 0.5).astype(int)

result_graph["predict"] = predict_edge_label
predict_grn = result_graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)
predict_grn


,Unnamed: 0,u,i,ts,label,idx,predict,source,target
0,0,1643,1,0.103799,1,1,0.054033,chr12-53370831-53371774,AAAS
1,1,1643,1,0.130393,1,2,1.000000,chr12-53370831-53371774,AAAS
2,2,1643,1,0.427715,1,3,1.000000,chr12-53370831-53371774,AAAS
3,3,1643,1,0.487691,1,4,1.000000,chr12-53370831-53371774,AAAS
4,4,1643,1,0.560851,1,5,1.000000,chr12-53370831-53371774,AAAS
...,...,...,...,...,...,...,...,...,...
1029875,1029875,5500,5500,10.258387,1,1029876,1.000000,chrX-7147230-7148785,chrX-7147230-7148785
1029876,1029876,5500,5500,10.318362,1,1029877,1.000000,chrX-7147230-7148785,chrX-7147230-7148785
1029877,1029877,5500,5500,10.318362,1,1029878,1.000000,chrX-7147230-7148785,chrX-7147230-7148785
1029878,1029878,5500,5500,10.552336,1,1029879,1.000000,chrX-7147230-7148785,chrX-7147230-7148785


In [34]:
peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()
print(peak_gene_df)


                          Peak    Gene         ts   predict
0      chr12-53370831-53371774    AAAS   0.103799  0.054033
1      chr12-53370831-53371774    AAAS   0.130393  1.000000
2      chr12-53370831-53371774    AAAS   0.427715  1.000000
3      chr12-53370831-53371774    AAAS   0.487691  1.000000
4      chr12-53370831-53371774    AAAS   0.560851  1.000000
...                        ...     ...        ...       ...
37223  chr20-45933784-45935583  ZSWIM1   9.705171  1.000000
37224  chr20-45933784-45935583  ZSWIM1   9.757383  1.000000
37225  chr20-45933784-45935583  ZSWIM1  10.113001  1.000000
37226  chr20-45933784-45935583  ZSWIM1  10.506163  1.000000
37227  chr20-45933784-45935583  ZSWIM1  10.729193  1.000000

[37228 rows x 4 columns]


## TF-gene network

In [14]:
merged_df = pd.merge(tf_peak_df, peak_gene_df, on='Peak')
print(merged_df)

tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
    avg_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

# 查看结果
print(tf_gene_grn)

condition = ~(tf_gene_grn['Gene'].str.startswith('chr'))
tf_gene_grn = tf_gene_grn[condition].copy()
condition2 = ~(peak_gene_df['Gene'].str.startswith('chr'))
peak_gene_df = peak_gene_df[condition2].copy()


                               Peak         TF  value  \
0                chr1-629315-630015  FOSB::JUN      1   
1                chr1-629315-630015  FOSB::JUN      1   
2                chr1-629315-630015  FOSB::JUN      1   
3                chr1-629315-630015  FOSB::JUN      1   
4                chr1-629315-630015  FOSB::JUN      1   
...                             ...        ...    ...   
816431229  chrX-155026675-155027933      PRDM9      1   
816431230  chrX-155026675-155027933      PRDM9      1   
816431231  chrX-155026675-155027933      PRDM9      1   
816431232  chrX-155026675-155027933      PRDM9      1   
816431233  chrX-155026675-155027933      PRDM9      1   

                               Gene        ts   predict  
0                          MTCO1P12  0.000000  0.064111  
1                          MTCO1P12  0.103799  0.999998  
2                          MTCO1P12  0.130393  1.000000  
3                          MTCO1P12  0.162789  1.000000  
4                        

In [35]:

print(f"TF-Peak: {tf_peak_df['TF'].nunique()}, {tf_peak_df['Peak'].nunique()}, edge: {len(tf_peak_df)}")
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")

TF-Peak: 828, 72563, edge: 55056390
Peak-Gene: 412, 246, edge:37228
TF-Gene: 827, 246, edge: 21678835


### save data

In [28]:
tf_gene_grn.to_pickle(dyg_result_path + "new_tf_gene_grn_1223.pkl")

# Average GRN


In [15]:
tf_gene_grn = pd.read_pickle(dyg_result_path + "new_tf_gene_grn_1223.pkl")
tf_gene_grn

In [29]:
tf_gene_grn

,TF,Gene,ts,peak_num,avg_weight,total_weight
0,ALX3,AAAS,0.000000,1,0.064111,0.064111
1,ALX3,AAAS,0.103799,2,0.527016,1.054032
2,ALX3,AAAS,0.130393,2,1.000000,2.000000
3,ALX3,AAAS,0.427715,2,1.000000,2.000000
4,ALX3,AAAS,0.487691,2,1.000000,2.000000
...,...,...,...,...,...,...
328801431,mix-a,ZSWIM1,10.038790,1,1.000000,1.000000
328801432,mix-a,ZSWIM1,10.113001,1,1.000000,1.000000
328801433,mix-a,ZSWIM1,10.506163,1,1.000000,1.000000
328801434,mix-a,ZSWIM1,10.584224,1,1.000000,1.000000


In [30]:
avg_active_tf_gene_grn = tf_gene_grn.groupby(['TF', 'Gene']).agg(
    avg_ts_weight=('total_weight', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('total_weight', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_tf_gene_grn.to_pickle(dyg_result_path + "average_active_tf_gene_grn.pkl")

print(avg_active_tf_gene_grn)


pivoted_grn = tf_gene_grn.pivot_table(
    index=['TF', 'Gene'],
    columns='ts',
    values='total_weight',
    fill_value=0
)
pivoted_grn['average_active_weight'] = pivoted_grn.mean(axis=1)
avg_global_tf_gene_grn = pivoted_grn.reset_index()
avg_global_tf_gene_grn = avg_global_tf_gene_grn[["TF","Gene","average_active_weight"]].copy()
avg_global_tf_gene_grn.columns.name = None
avg_global_tf_gene_grn.to_pickle(dyg_result_path + "average_global_tf_gene_grn.pkl")

print(avg_global_tf_gene_grn)

print(f"TF-Gene: {avg_active_tf_gene_grn['TF'].nunique()}, {avg_active_tf_gene_grn['Gene'].nunique()}, edge: {len(avg_active_tf_gene_grn)}")

     TF      Gene  avg_ts_weight  avg_total_weight
0  ALX3      AAAS       1.333969        248.118149
1  ALX3  AASDHPPT       0.987846         76.064110
2  ALX3     ABHD6       0.983581         56.064110
3  ALX3      ACO2       1.557796        288.192322
4  ALX3     ADCY3       0.989601         89.064110
           TF      Gene  average_active_weight
0        ALX3      AAAS               1.112637
1        ALX3  AASDHPPT               0.341095
2        ALX3     ABHD6               0.251409
3        ALX3      ACO2               1.292342
4        ALX3     ADCY3               0.399391
...       ...       ...                    ...
189322  mix-a    ZNF473               0.368000
189323  mix-a    ZNF581               1.552309
189324  mix-a    ZNF598               0.381408
189325  mix-a    ZNF672               0.848292
189326  mix-a    ZSWIM1               0.798781

[189327 rows x 3 columns]


# Total Code

In [36]:
graph_df = pd.read_pickle(dyg_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']


result_path = dyg_result_path + 'my_result_run{run}.npy'
predict_edge_label = np.load(result_path)
New_Graph["predict"] = predict_edge_label
predict_grn = New_Graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)
peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()
print("*"*20)
print(peak_gene_df)


jaspar_tf_region_file = org_data_path + "jaspar_data.h5ad"
jaspar_data = ad.read_h5ad(jaspar_tf_region_file)
adata_region_tf = filter_jaspar_tf(jaspar_data)

coo_matrix = adata_region_tf.X.tocoo()
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})


merged_df = pd.merge(tf_peak_df, peak_gene_df, on='Peak')
print("*"*50)
print(merged_df)

tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
    avg_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

# 查看结果
print("*"*50)
print(tf_gene_grn)

tf_gene_grn.to_pickle(dyg_result_path + "new_tf_gene_grn.pkl")


avg_active_tf_gene_grn = tf_gene_grn.groupby(['TF', 'Gene']).agg(
    avg_ts_weight=('total_weight', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('total_weight', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_tf_gene_grn.to_pickle(dyg_result_path + "average_active_tf_gene_grn.pkl")

print("*"*50)
print(avg_active_tf_gene_grn)


pivoted_grn = tf_gene_grn.pivot_table(
    index=['TF', 'Gene'],
    columns='ts',
    values='total_weight',
    fill_value=0
)
pivoted_grn['average_active_weight'] = pivoted_grn.mean(axis=1)
avg_global_tf_gene_grn = pivoted_grn.reset_index()
avg_global_tf_gene_grn = avg_global_tf_gene_grn[["TF","Gene","average_active_weight"]].copy()
avg_global_tf_gene_grn.columns.name = None
avg_global_tf_gene_grn.to_pickle(dyg_result_path + "average_global_tf_gene_grn.pkl")

print("*"*50)
print(avg_global_tf_gene_grn)

print("*"*50)
print(f"TF-Peak: {tf_peak_df['TF'].nunique()}, {tf_peak_df['Peak'].nunique()}, edge: {len(tf_peak_df)}")
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")
print(f"TF-Gene: {avg_active_tf_gene_grn['TF'].nunique()}, {avg_active_tf_gene_grn['Gene'].nunique()}, edge: {len(avg_active_tf_gene_grn)}")

********************
                          Peak    Gene         ts   predict
0      chr12-53370831-53371774    AAAS   0.103799  0.054033
1      chr12-53370831-53371774    AAAS   0.130393  1.000000
2      chr12-53370831-53371774    AAAS   0.427715  1.000000
3      chr12-53370831-53371774    AAAS   0.487691  1.000000
4      chr12-53370831-53371774    AAAS   0.560851  1.000000
...                        ...     ...        ...       ...
37223  chr20-45933784-45935583  ZSWIM1   9.705171  1.000000
37224  chr20-45933784-45935583  ZSWIM1   9.757383  1.000000
37225  chr20-45933784-45935583  ZSWIM1  10.113001  1.000000
37226  chr20-45933784-45935583  ZSWIM1  10.506163  1.000000
37227  chr20-45933784-45935583  ZSWIM1  10.729193  1.000000

[37228 rows x 4 columns]


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 1: 过滤 Peaks (行)
  > 找到 72563 / 72584 个 peaks 至少有 1 个 TF 结合。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 2: 过滤 TFs (列)
  > 找到 879 / 879 个 TFs 至少结合 1 个 peak。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  > 最终形状: (72563, 879)
********************
                              Peak         TF  value      Gene         ts  \
0               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.000000   
1               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.103799   
2               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.130393   
3               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.162789   
4               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.306248   
...                            ...        ...    ...       ...        ...   
29518733  chrX-153926220-153928652      PRDM9      1     HCFC1   9.813189   
29518734  chrX-153926220-153928652      PRDM9      1     HCFC1  10.318362   
29518735  chrX-153926220-153928652      PRDM9      1     HCFC1  10.506163   
29518736  chrX-153926220-153928652      PRDM9      1     HCFC1  10.584224   
29518737  chrX-153926220-153928652      PRDM9      1     HCFC1  10.638032   

           predict  
0         

# Further

In [45]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/original/"
adata_rp_gene_peak = ad.read_h5ad(output_path + "binary_peak_gene_rp_network.h5ad")

In [46]:
print(adata_rp_gene_peak)
from data_preprocess import adata_to_dataframe

prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)

prior_peak_gene_df.head()
prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})
print(prior_peak_gene_df)

AnnData object with n_obs × n_vars = 2000 × 71541
    uns: 'decay_distance', 'description', 'max_range'
       Gene                      Peak  value
0      AAAS   chr12-53251618-53252739      1
1      AAAS   chr12-53267768-53268827      1
2      AAAS   chr12-53295185-53295894      1
3      AAAS   chr12-53299560-53300133      1
4      AAAS   chr12-53336297-53336656      1
...     ...                       ...    ...
12237  ZXDC  chr3-126522381-126522675      1
12238   ZYX  chr7-143327917-143328230      1
12239   ZYX  chr7-143362369-143362688      1
12240   ZYX  chr7-143380273-143381719      1
12241   ZYX  chr7-143408952-143409231      1

[12242 rows x 3 columns]


In [49]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/dygmamba/res/result2/"

adata_rp_gene_peak = ad.read_h5ad(output_path + "rp_gene_peak.h5ad")
print(adata_rp_gene_peak)
prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)

prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})
print(prior_peak_gene_df)


AnnData object with n_obs × n_vars = 500 × 5000
    uns: 'decay_distance', 'description', 'max_range'
        Gene                       Peak  value
0     NDUFS5     chr1-38990697-38992620      1
1       DPP9      chr19-4790997-4792145      1
2    TXNDC15   chr5-134904464-134905833      1
3      PPRC1  chr10-102055526-102056382      1
4      PPRC1  chr10-102064962-102066074      1
..       ...                        ...    ...
437     ELF2   chr4-139176167-139178479      1
438      IVD    chr15-40440044-40441714      1
439    CHTOP   chr1-153670613-153672079      1
440    KIF3A   chr5-132662988-132664501      1
441    KIF3A   chr5-132674425-132675608      1

[442 rows x 3 columns]
